In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# =========================================================
# DATABASE GOLD
# =========================================================

spark.sql("CREATE DATABASE IF NOT EXISTS gold")

# =========================================================
# DIM_CLIENTE
# =========================================================

clientes = spark.table("silver.clientes")
enderecos = spark.table("silver.enderecos_clientes")

dim_cliente = (
    clientes.alias("c")
    .join(
        enderecos.alias("e"),
        (
            (col("c.id_cliente") == col("e.id_cliente")) &
            col("e.principal_entrega")
        ),
        "left"
    )
    .select(
        monotonically_increasing_id().alias("sk_cliente"),

        col("c.id_cliente").alias("id_cliente_origem"),

        col("c.nome_cliente"),
        col("c.sexo"),
        col("c.data_nascimento"),

        floor(months_between(current_date(), col("c.data_nascimento")) / 12)
            .alias("idade"),

        when(
            floor(months_between(current_date(), col("c.data_nascimento")) / 12) < 25,
            "18-24"
        )
        .when(
            floor(months_between(current_date(), col("c.data_nascimento")) / 12) < 35,
            "25-34"
        )
        .when(
            floor(months_between(current_date(), col("c.data_nascimento")) / 12) < 45,
            "35-44"
        )
        .when(
            floor(months_between(current_date(), col("c.data_nascimento")) / 12) < 60,
            "45-59"
        )
        .otherwise("60+")
        .alias("faixa_etaria"),

        col("c.renda_mensal"),

        when(col("c.renda_mensal") < 3000, "Baixa")
        .when(col("c.renda_mensal") < 8000, "Media")
        .when(col("c.renda_mensal") < 15000, "Alta")
        .otherwise("Premium")
        .alias("faixa_renda"),

        col("e.cidade"),
        col("e.estado"),

        when(col("e.estado").isin("SP","RJ","MG","ES"), "Sudeste")
        .when(col("e.estado").isin("RS","SC","PR"), "Sul")
        .when(col("e.estado").isin("BA","PE","CE"), "Nordeste")
        .when(col("e.estado").isin("GO","MT","MS","DF"), "Centro-Oeste")
        .otherwise("Outros")
        .alias("regiao"),

        col("c.data_cadastro")
    )
)

dim_cliente.write.format("delta").mode("overwrite").saveAsTable("gold.dim_cliente")

# =========================================================
# DIM_PRODUTO
# =========================================================

produtos = spark.table("silver.produtos")
subcategorias = spark.table("silver.subcategorias")
categorias = spark.table("silver.categorias")

dim_produto = (
    produtos.alias("p")
    .join(
        subcategorias.alias("s"),
        col("p.id_subcategoria") == col("s.id_subcategoria")
    )
    .join(
        categorias.alias("c"),
        col("s.id_categoria") == col("c.id_categoria")
    )
    .select(
        monotonically_increasing_id().alias("sk_produto"),

        col("p.id_produto").alias("id_produto_origem"),

        col("p.nome_produto"),
        col("p.marca"),

        col("c.nome_categoria").alias("categoria"),
        col("s.nome_subcategoria").alias("subcategoria"),

        col("p.preco").alias("preco_base"),

        when(col("p.preco") < 100, "Ate 100")
        .when(col("p.preco") < 500, "100-499")
        .when(col("p.preco") < 2000, "500-1999")
        .when(col("p.preco") < 5000, "2000-4999")
        .otherwise("5000+")
        .alias("faixa_preco"),

        col("p.ativo")
    )
)

dim_produto.write.format("delta").mode("overwrite").saveAsTable("gold.dim_produto")

# =========================================================
# DIM_VENDEDOR
# =========================================================

vendedores = spark.table("silver.vendedores")

dim_vendedor = (
    vendedores
    .select(
        monotonically_increasing_id().alias("sk_vendedor"),

        col("id_vendedor").alias("id_vendedor_origem"),

        col("nome_vendedor"),
        col("cidade"),
        col("estado"),

        when(col("estado").isin("SP","RJ","MG","ES"), "Sudeste")
        .when(col("estado").isin("RS","SC","PR"), "Sul")
        .when(col("estado").isin("BA","PE","CE"), "Nordeste")
        .when(col("estado").isin("GO","MT","MS","DF"), "Centro-Oeste")
        .otherwise("Outros")
        .alias("regiao"),

        col("data_entrada")
    )
)

dim_vendedor.write.format("delta").mode("overwrite").saveAsTable("gold.dim_vendedor")

# =========================================================
# DIM_PAGAMENTO
# =========================================================

pagamentos = spark.table("silver.pagamentos")

dim_pagamento = (
    pagamentos
    .select(
        col("tipo_pagamento"),

        when(col("parcelas") == 1, "1x")
        .when(col("parcelas") <= 3, "2-3x")
        .when(col("parcelas") <= 6, "4-6x")
        .otherwise("7+x")
        .alias("faixa_parcelas")
    )
    .distinct()
    .withColumn(
        "sk_pagamento",
        monotonically_increasing_id()
    )
    .select(
        "sk_pagamento",
        "tipo_pagamento",
        "faixa_parcelas"
    )
)

dim_pagamento.write.format("delta").mode("overwrite").saveAsTable("gold.dim_pagamento")

# =========================================================
# DIM_ENTREGA
# =========================================================

entregas = spark.table("silver.entregas")

dim_entrega = (
    entregas
    .select(
        col("tipo_entrega"),

        when(
            datediff(col("data_entrega_real"), col("data_envio")) <= 3,
            "Ate 3 dias"
        )
        .when(
            datediff(col("data_entrega_real"), col("data_envio")) <= 7,
            "4-7 dias"
        )
        .when(
            datediff(col("data_entrega_real"), col("data_envio")) <= 15,
            "8-15 dias"
        )
        .otherwise("15+ dias")
        .alias("faixa_prazo"),

        when(
            col("data_entrega_real") > col("data_entrega_prevista"),
            "Sim"
        )
        .otherwise("Nao")
        .alias("entrega_atrasada")
    )
    .distinct()
    .withColumn(
        "sk_entrega",
        monotonically_increasing_id()
    )
    .select(
        "sk_entrega",
        "tipo_entrega",
        "faixa_prazo",
        "entrega_atrasada"
    )
)

dim_entrega.write.format("delta").mode("overwrite").saveAsTable("gold.dim_entrega")

# =========================================================
# DIM_DATA
# =========================================================

datas = spark.sql("""
SELECT explode(
    sequence(
        to_date('2022-01-01'),
        to_date('2030-12-31'),
        interval 1 day
    )
) AS data_ref
""")

dim_data = (
    datas
    .select(
        date_format(col("data_ref"), "yyyyMMdd")
            .cast("int")
            .alias("sk_data"),

        col("data_ref").alias("data_completa"),

        dayofmonth(col("data_ref")).alias("dia"),

        date_format(col("data_ref"), "EEEE")
            .alias("nome_dia_semana"),

        weekofyear(col("data_ref")).alias("semana_ano"),

        month(col("data_ref")).alias("mes"),

        date_format(col("data_ref"), "MMMM")
            .alias("nome_mes"),

        quarter(col("data_ref")).alias("trimestre"),

        when(month(col("data_ref")) <= 6, 1)
        .otherwise(2)
        .alias("semestre"),

        year(col("data_ref")).alias("ano"),

        when(
            dayofweek(col("data_ref")).isin(1,7),
            1
        )
        .otherwise(0)
        .alias("fim_semana")
    )
)

dim_data.write.format("delta").mode("overwrite").saveAsTable("gold.dim_data")

# =========================================================
# FATO_VENDAS
# =========================================================

itens = spark.table("silver.itens_pedido")
pedidos = spark.table("silver.pedidos")
produtos = spark.table("silver.produtos")
pagamentos = spark.table("silver.pagamentos")
entregas = spark.table("silver.entregas")
avaliacoes = spark.table("silver.avaliacoes")

dim_cliente = spark.table("gold.dim_cliente")
dim_produto = spark.table("gold.dim_produto")
dim_vendedor = spark.table("gold.dim_vendedor")
dim_pagamento = spark.table("gold.dim_pagamento")
dim_entrega = spark.table("gold.dim_entrega")

fato_vendas = (
    itens.alias("ip")

    .join(
        pedidos.alias("pe"),
        col("ip.id_pedido") == col("pe.id_pedido")
    )

    .join(
        produtos.alias("pr"),
        col("ip.id_produto") == col("pr.id_produto")
    )

    .join(
        pagamentos.alias("pg"),
        col("pe.id_pedido") == col("pg.id_pedido")
    )

    .join(
        entregas.alias("en"),
        col("pe.id_pedido") == col("en.id_pedido")
    )

    .join(
        avaliacoes.alias("av"),
        col("pe.id_pedido") == col("av.id_pedido"),
        "left"
    )

    .join(
        dim_cliente.alias("dc"),
        col("pe.id_cliente") == col("dc.id_cliente_origem")
    )

    .join(
        dim_produto.alias("dp"),
        col("ip.id_produto") == col("dp.id_produto_origem")
    )

    .join(
        dim_vendedor.alias("dv"),
        col("ip.id_vendedor") == col("dv.id_vendedor_origem")
    )

    .withColumn(
        "faixa_parcelas_calc",
        when(col("pg.parcelas") == 1, "1x")
        .when(col("pg.parcelas") <= 3, "2-3x")
        .when(col("pg.parcelas") <= 6, "4-6x")
        .otherwise("7+x")
    )

    .join(
        dim_pagamento.alias("dpg"),
        (
            (col("pg.tipo_pagamento") == col("dpg.tipo_pagamento")) &
            (col("faixa_parcelas_calc") == col("dpg.faixa_parcelas"))
        )
    )

    .withColumn(
        "faixa_prazo_calc",
        when(
            datediff(col("en.data_entrega_real"), col("en.data_envio")) <= 3,
            "Ate 3 dias"
        )
        .when(
            datediff(col("en.data_entrega_real"), col("en.data_envio")) <= 7,
            "4-7 dias"
        )
        .when(
            datediff(col("en.data_entrega_real"), col("en.data_envio")) <= 15,
            "8-15 dias"
        )
        .otherwise("15+ dias")
    )

    .withColumn(
        "entrega_atrasada_calc",
        when(
            col("en.data_entrega_real") > col("en.data_entrega_prevista"),
            "Sim"
        )
        .otherwise("Nao")
    )

    .join(
        dim_entrega.alias("de"),
        (
            (col("en.tipo_entrega") == col("de.tipo_entrega")) &
            (col("faixa_prazo_calc") == col("de.faixa_prazo")) &
            (col("entrega_atrasada_calc") == col("de.entrega_atrasada"))
        )
    )

    .select(
        col("dc.sk_cliente"),
        col("dp.sk_produto"),
        col("dv.sk_vendedor"),
        col("dpg.sk_pagamento"),
        col("de.sk_entrega"),

        date_format(col("pe.data_pedido"), "yyyyMMdd")
            .cast("int")
            .alias("sk_data"),

        col("pe.id_pedido").alias("id_pedido_origem"),

        col("ip.quantidade").alias("quantidade_itens"),

        col("ip.preco_unitario"),

        (col("ip.quantidade") * col("ip.preco_unitario"))
            .alias("valor_bruto"),

        col("ip.desconto"),

        col("en.valor_frete"),

        (
            (col("ip.quantidade") * col("ip.preco_unitario"))
            - col("ip.desconto")
            + col("en.valor_frete")
        ).alias("valor_liquido"),

        (col("pr.custo") * col("ip.quantidade"))
            .alias("custo_produto"),

        (
            (
                (col("ip.quantidade") * col("ip.preco_unitario"))
                - col("ip.desconto")
                + col("en.valor_frete")
            )
            -
            (col("pr.custo") * col("ip.quantidade"))
        ).alias("lucro"),

        datediff(
            col("en.data_entrega_real"),
            col("en.data_envio")
        ).alias("prazo_entrega_dias"),

        col("av.nota").alias("avaliacao")
    )
)

fato_vendas.write.format("delta").mode("overwrite").saveAsTable("gold.fato_vendas")

print("Camada GOLD criada com sucesso.")

In [0]:
%sql
select * from gold.fato_vendas limit 10